# 💘 ¿Habrá Match? - Modelamiento

## FASE 4 - MODELAMIENTO, EVALUACIÓN E INTERPRETACIÓN

## 4.1 Setup y carga de datos

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import time
import joblib
warnings.filterwarnings('ignore')

X_train = pd.read_csv('../data/X_train.csv')
X_test = pd.read_csv('../data/X_test.csv')
y_train = pd.read_csv('../data/y_train.csv').values.ravel()
y_test = pd.read_csv('../data/y_test.csv').values.ravel()

print(f'Train: {X_train.shape}, Test: {X_test.shape}')
print(f'Train match rate: {y_train.mean():.3f}')
print(f'Test match rate: {y_test.mean():.3f}')

## 4.2 Función de evaluación

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

def evaluate_model(name, model, X_tr, y_tr, X_te, y_te):
    model.fit(X_tr, y_tr)
    y_pred = model.predict(X_te)
    y_prob = model.predict_proba(X_te)[:, 1] if hasattr(model, 'predict_proba') else model.decision_function(X_te)
    
    return {
        'Modelo': name,
        'Accuracy': round(accuracy_score(y_te, y_pred), 4),
        'Precision': round(precision_score(y_te, y_pred, zero_division=0), 4),
        'Recall': round(recall_score(y_te, y_pred, zero_division=0), 4),
        'F1': round(f1_score(y_te, y_pred, zero_division=0), 4),
        'ROC-AUC': round(roc_auc_score(y_te, y_prob), 4)
    }, model, y_prob

## 4.3 Configurar modelos


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier

modelos = {
    'Random Forest': RandomForestClassifier(n_estimators=200, random_state=42, max_depth=10),
    'XGBoost': XGBClassifier(n_estimators=200, use_label_encoder=False, eval_metric='logloss', random_state=42, n_jobs=-1),
    'Decision Tree': DecisionTreeClassifier(random_state=42, max_depth=8),
    'SVM': SVC(probability=True, random_state=42),
    'KNN': KNeighborsClassifier(n_neighbors=7),
    'MLP': MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=300, random_state=42)
}

## 4.4 Entrenar y evaluar

In [ ]:
resultados = []
probs_dict = {}
modelos_entrenados = {}

for nombre, modelo in modelos.items():
    print(f'Entrenando: {nombre}...')
    metricas, modelo_fit, y_prob = evaluate_model(nombre, modelo, X_train, y_train, X_test, y_test)
    resultados.append(metricas)
    probs_dict[nombre] = y_prob
    modelos_entrenados[nombre] = modelo_fit
    print(f'  ROC-AUC: {metricas["ROC-AUC"]}')

df_resultados = pd.DataFrame(resultados).sort_values('ROC-AUC', ascending=False)
df_resultados

## 4.5 Guardar mejor modelo

In [ ]:
mejor_nombre = df_resultados.iloc[0]['Modelo']
mejor_modelo = modelos_entrenados[mejor_nombre]

from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

scaler = StandardScaler()
pipeline = Pipeline([('scaler', scaler), ('model', mejor_modelo)])
pipeline.fit(X_train, y_train)

import os
os.makedirs('../models', exist_ok=True)

joblib.dump(pipeline, '../models/pipeline_match_predictor.pkl')
joblib.dump(list(X_train.columns), '../models/feature_names.pkl')
joblib.dump({'ROC-AUC': df_resultados.iloc[0]['ROC-AUC'],
             'Accuracy': df_resultados.iloc[0]['Accuracy'],
             'F1': df_resultados.iloc[0]['F1'],
             'Precision': df_resultados.iloc[0]['Precision'],
             'Recall': df_resultados.iloc[0]['Recall']},
            '../models/metricas_finales.pkl')

print(f'Mejor modelo guardado: {mejor_nombre}')
print(f'ROC-AUC: {df_resultados.iloc[0]["ROC-AUC"]}')

## 4.6 Visualizaciones

In [ ]:
import matplotlib.pyplot as plt
metricas_plot = ['Accuracy', 'Precision', 'Recall', 'F1', 'ROC-AUC']
fig, axes = plt.subplots(1, 5, figsize=(20, 4))
colors = sns.color_palette('RdPu', len(modelos))

for i, metrica in enumerate(metricas_plot):
    vals = df_resultados[metrica].values
    names = [n[:12] for n in df_resultados['Modelo'].values]
    axes[i].bar(names, vals, color=colors)
    axes[i].set_title(metrica, fontweight='bold')
    axes[i].set_ylim(0, 1.1)

plt.suptitle('Comparación de Métricas - 7 Modelos', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/comparacion_metricas.png', dpi=150, bbox_inches='tight')
plt.show()